In [ ]:
"""
PURPOSE: To transform the SQL data mart produced by Notebook 2 into a modeling-ready
dataset by adding domain-informed healthcare assumptions and market-access
scenario variables.

This notebook deliberately separates:

1. Observed data
2. External/domain-informed assumptions
3. Synthetic scenario variables
4. Derived features
5. Target-derived descriptive metrics

KEY PRINCIPLES
--------------
- Notebook 2 owns ETL and data modeling.
- Notebook 3 owns domain enrichment and feature governance.
- No target-derived variable enters the modeling dataset.
- Every feature has provenance.
- Every modeling feature has a forecast-time availability assessment.
- The underlying sales dataset is synthetic.
- Domain assumptions are NOT presented as observed real-world market data.

OUTPUTS
-------
data/processed/modeling_dataset.csv
data/processed/feature_provenance.csv
data/processed/feature_availability_audit.csv
data/processed/descriptive_uptake_analysis.csv
================================================================================
"""

# ==============================================================================
# 1. SETUP
# ==============================================================================

from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# 2. LOAD SQL DATA MART
# ==============================================================================

DATA_MART_FILE = "fact_drug_region_month.csv"

df = pd.read_csv(DATA_MART_FILE)
df["month"] = pd.to_datetime(df["month"])

df = df.sort_values(
    ["drug_id", "region", "month"]
).reset_index(drop=True)

print("\nData mart loaded")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date range:", df["month"].min().date(), "to", df["month"].max().date())

display(df.head())

In [ ]:
# ==============================================================================
# 3. DATA CONTRACT VALIDATION
# ==============================================================================

required_columns = [
    "drug_id",
    "category",
    "region",
    "month",
    "utilization",
    "months_since_launch"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# One row per drug-region-month
duplicate_count = df.duplicated(
    ["drug_id", "region", "month"]
).sum()

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate drug-region-month rows."
)

# Target validity
assert df["utilization"].notna().all()
assert (df["utilization"] >= 0).all()

# Expected current dataset structure
n_series = df.groupby(
    ["drug_id", "region"]
).ngroups

series_month_counts = df.groupby(
    ["drug_id", "region"]
)["month"].nunique()

print("\nDATA CONTRACT")
print("-" * 60)
print("Series:", n_series)
print("Minimum months per series:", series_month_counts.min())
print("Maximum months per series:", series_month_counts.max())
print("Duplicate drug-region-month rows:", duplicate_count)

assert series_month_counts.min() == series_month_counts.max()

In [ ]:
# ==============================================================================
# 4. DOMAIN-INFORMED REGION ASSUMPTIONS
# ==============================================================================

region_assumptions = pd.DataFrame({
    "region": [
        "Africa",
        "Europe",
        "East Asia",
        "North America",
        "South Asia",
        "Middle East",
        "South America",
        "Oceania"
    ],

    "population_millions": [
        1460, 745, 1680, 375,
        1970, 470, 435, 45
    ],

    "chronic_prevalence_pct": [
        28.5, 40.0, 29.0, 32.0,
        27.0, 38.0, 30.0, 29.0
    ],

    "cough_cold_episodes_per_year": [
        1.4, 2.0, 1.8, 2.3,
        1.5, 1.9, 1.8, 2.1
    ],

    "antibiotic_episodes_per_year": [
        0.4, 0.5, 0.5, 0.6,
        0.4, 0.5, 0.5, 0.5
    ],

    "antipyretic_episodes_per_year": [
        0.8, 1.0, 1.0, 1.1,
        0.9, 1.0, 1.0, 1.0
    ],

    "vitamin_usage_pct": [
        15, 40, 30, 55,
        12, 25, 25, 35
    ]
})

assert set(region_assumptions["region"]) == set(df["region"].unique())

print("Domain assumption table:")
display(region_assumptions)

In [ ]:
# ==============================================================================
# 5. FEATURE PROVENANCE
# ==============================================================================

provenance_table = pd.DataFrame([
    {
        "feature": "utilization",
        "provenance_type": "Observed",
        "source": "Synthetic pharmacy sales dataset",
        "forecast_use": "Target",
        "modeling_status": "Target"
    },
    {
        "feature": "population_millions",
        "provenance_type": "Domain assumption",
        "source": "Population-scale assumption; external-data structure",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "chronic_prevalence_pct",
        "provenance_type": "Domain assumption",
        "source": "Literature-informed / estimated",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "cough_cold_episodes_per_year",
        "provenance_type": "Domain assumption",
        "source": "Literature-consistent estimate",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "antibiotic_episodes_per_year",
        "provenance_type": "Domain assumption",
        "source": "Literature-consistent estimate",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "antipyretic_episodes_per_year",
        "provenance_type": "Domain assumption",
        "source": "Literature-consistent estimate",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "vitamin_usage_pct",
        "provenance_type": "Domain assumption",
        "source": "Reasoned estimate",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "reimbursement_tier",
        "provenance_type": "Synthetic scenario",
        "source": "Randomized demonstration variable, seed=42",
        "forecast_use": "Assumed known",
        "modeling_status": "Use with caveat"
    },
    {
        "feature": "competitor_entry_month_offset",
        "provenance_type": "Synthetic scenario",
        "source": "Randomized demonstration variable, seed=42",
        "forecast_use": "Assumed known",
        "modeling_status": "Use with caveat"
    },
    {
        "feature": "competitor_present",
        "provenance_type": "Derived synthetic scenario",
        "source": "Derived from synthetic competitor timing",
        "forecast_use": "Assumed known",
        "modeling_status": "Use with caveat"
    },
    {
        "feature": "eligible_population_proxy",
        "provenance_type": "Derived",
        "source": "Population × domain rate/assumption",
        "forecast_use": "Known/static assumption",
        "modeling_status": "Use"
    },
    {
        "feature": "uptake_rate_proxy",
        "provenance_type": "Target-derived",
        "source": "Utilization / eligible population",
        "forecast_use": "Not independently known",
        "modeling_status": "EXCLUDE"
    }
])

display(provenance_table)

In [ ]:
# ==============================================================================
# 6. FORECAST-TIME FEATURE AVAILABILITY AUDIT
# ==============================================================================

feature_availability_audit = pd.DataFrame([
    {
        "feature": "utilization_lag1",
        "information_class": "Historical",
        "known_at_forecast_origin": "Yes",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "utilization_lag12",
        "information_class": "Historical",
        "known_at_forecast_origin": "Yes",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "month_of_year",
        "information_class": "Calendar",
        "known_at_forecast_origin": "Yes",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "months_since_launch",
        "information_class": "Derived calendar",
        "known_at_forecast_origin": "Yes, using launch proxy",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "population_millions",
        "information_class": "Domain assumption",
        "known_at_forecast_origin": "Yes",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "eligible_population_proxy",
        "information_class": "Derived domain assumption",
        "known_at_forecast_origin": "Yes",
        "leakage_risk": "Low",
        "modeling_status": "Use"
    },
    {
        "feature": "reimbursement_tier",
        "information_class": "Synthetic scenario",
        "known_at_forecast_origin": "Assumed",
        "leakage_risk": "Scenario-dependent",
        "modeling_status": "Use with caveat"
    },
    {
        "feature": "competitor_present",
        "information_class": "Synthetic scenario",
        "known_at_forecast_origin": "Assumed",
        "leakage_risk": "Scenario-dependent",
        "modeling_status": "Use with caveat"
    },
    {
        "feature": "uptake_rate_proxy",
        "information_class": "Target-derived",
        "known_at_forecast_origin": "No",
        "leakage_risk": "HIGH — target leakage",
        "modeling_status": "EXCLUDE"
    },
    {
        "feature": "future_utilization",
        "information_class": "Target",
        "known_at_forecast_origin": "No",
        "leakage_risk": "HIGH — target leakage",
        "modeling_status": "EXCLUDE"
    }
])

display(feature_availability_audit)

In [ ]:
# ==============================================================================
# 7. JOIN DOMAIN ASSUMPTIONS
# ==============================================================================

df = df.merge(
    region_assumptions,
    on="region",
    how="left",
    validate="many_to_one"
)

assert df[
    [
        "population_millions",
        "chronic_prevalence_pct",
        "cough_cold_episodes_per_year",
        "antibiotic_episodes_per_year",
        "antipyretic_episodes_per_year",
        "vitamin_usage_pct"
    ]
].notna().all().all()



In [ ]:
# ==============================================================================
# 8. SYNTHETIC MARKET-ACCESS SCENARIOS
# ==============================================================================

np.random.seed(42)

drugs = sorted(df["drug_id"].unique())

reimbursement_tiers = pd.DataFrame({
    "drug_id": drugs,
    "reimbursement_tier": np.random.choice(
        ["High", "Medium", "Low"],
        size=len(drugs),
        p=[0.4, 0.4, 0.2]
    )
})

competitor_entry = pd.DataFrame({
    "drug_id": drugs,
    "competitor_entry_month_offset": np.random.randint(
        6, 30, size=len(drugs)
    )
})

df = df.merge(
    reimbursement_tiers,
    on="drug_id",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    competitor_entry,
    on="drug_id",
    how="left",
    validate="many_to_one"
)

df["competitor_present"] = (
    df["months_since_launch"]
    >= df["competitor_entry_month_offset"]
).astype(int)



In [ ]:
# ==============================================================================
# 9. ELIGIBLE POPULATION PROXY
# ==============================================================================

df["eligible_population_proxy"] = np.select(
    [
        df["category"].eq("Chronic"),
        df["category"].eq("Cough_Cold"),
        df["category"].eq("Antibiotic"),
        df["category"].eq("Antipyretic"),
        df["category"].eq("Vitamin")
    ],
    [
        df["population_millions"] * 1_000_000
        * df["chronic_prevalence_pct"] / 100,

        df["population_millions"] * 1_000_000
        * df["cough_cold_episodes_per_year"],

        df["population_millions"] * 1_000_000
        * df["antibiotic_episodes_per_year"],

        df["population_millions"] * 1_000_000
        * df["antipyretic_episodes_per_year"],

        df["population_millions"] * 1_000_000
        * df["vitamin_usage_pct"] / 100
    ],
    default=np.nan
)

assert df["eligible_population_proxy"].notna().all()
assert (df["eligible_population_proxy"] > 0).all()


In [ ]:
# ==============================================================================
# 10. DESCRIPTIVE UPTAKE ANALYSIS
# IMPORTANT: NOT A MODELING FEATURE
# ==============================================================================

descriptive_uptake = df[
    [
        "drug_id",
        "category",
        "region",
        "month",
        "utilization",
        "eligible_population_proxy"
    ]
].copy()

descriptive_uptake["uptake_rate_proxy"] = (
    descriptive_uptake["utilization"]
    / descriptive_uptake["eligible_population_proxy"]
)

descriptive_uptake.to_csv(
    "descriptive_uptake_analysis.csv",
    index=False
)


In [ ]:
# ==============================================================================
# 11. PRODUCT AGE PROXY
# ==============================================================================

df["product_age_months"] = df["months_since_launch"]

print(
    "NOTE: product_age_months is based on first observed date "
    "and is therefore a launch proxy, not a confirmed commercial launch date."
)

In [ ]:
# ==============================================================================
# 12. FINAL MODELING DATASET
# ==============================================================================

modeling_columns = [
    "drug_id",
    "category",
    "region",
    "month",
    "utilization",
    "product_age_months",

    # Domain variables
    "population_millions",
    "chronic_prevalence_pct",
    "cough_cold_episodes_per_year",
    "antibiotic_episodes_per_year",
    "antipyretic_episodes_per_year",
    "vitamin_usage_pct",

    "eligible_population_proxy",

    # Synthetic scenario variables
    "reimbursement_tier",
    "competitor_entry_month_offset",
    "competitor_present"
]

modeling_dataset = df[modeling_columns].copy()

# Explicitly prove the target-derived variable is absent
assert "uptake_rate_proxy" not in modeling_dataset.columns

# Validate final grain
assert not modeling_dataset.duplicated(
    ["drug_id", "region", "month"]
).any()

assert modeling_dataset["utilization"].notna().all()

print("\nFINAL MODELING DATASET")
print("-" * 60)
print("Rows:", len(modeling_dataset))
print("Columns:", len(modeling_dataset.columns))
print("Unique drug-region series:",
      modeling_dataset.groupby(["drug_id", "region"]).ngroups)

display(modeling_dataset.head())

In [ ]:
# ==============================================================================
# 13. SAVE OUTPUTS
# ==============================================================================

modeling_dataset.to_csv(
    "modeling_dataset.csv",
    index=False
)

provenance_table.to_csv(
    "feature_provenance.csv",
    index=False
)

feature_availability_audit.to_csv(
    "feature_availability_audit.csv",
    index=False
)

descriptive_uptake.to_csv(
    "descriptive_uptake_analysis.csv",
    index=False
)

print("\n" + "=" * 80)
print("NOTEBOOK 3 OUTPUTS")
print("=" * 80)

print("✓ modeling_dataset.csv")
print("✓ feature_provenance.csv")
print("✓ feature_availability_audit.csv")
print("✓ descriptive_uptake_analysis.csv")

In [ ]:
# ==============================================================================
# 15. FINAL GOVERNANCE CHECK
# ==============================================================================

print("\nFINAL GOVERNANCE CHECK")
print("-" * 60)

checks = {
    "No duplicate drug-region-month rows":
        not modeling_dataset.duplicated(
            ["drug_id", "region", "month"]
        ).any(),

    "No missing target":
        modeling_dataset["utilization"].notna().all(),

    "No negative target":
        (modeling_dataset["utilization"] >= 0).all(),

    "Eligible population available":
        modeling_dataset["eligible_population_proxy"].notna().all(),

    "Target-derived uptake excluded":
        "uptake_rate_proxy" not in modeling_dataset.columns,

    "Provenance table created":
        len(provenance_table) > 0,

    "Feature availability audit created":
        len(feature_availability_audit) > 0
}

for check, result in checks.items():
    print(f"{'✓' if result else '✗'} {check}")

assert all(checks.values())

print("\n✓ NOTEBOOK 3 PASSED ALL GOVERNANCE CHECKS")